# Organizando a saída por Copa

## Task 2: organizar a saída por Copa

Quem consulta as partidas pergunta por **uma Copa** de cada vez: a tabela fica organizada **por Copa**.

In [1]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-particionamento")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o mesmo código do vídeo 5.1 (`ler_partidas` e o histórico).

In [2]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

A próxima célula também vem pronta: ela só apaga as tabelas que este notebook grava, para ele poder rodar de novo.

In [3]:
import shutil
for pasta in ["partidas_por_ano"]:
    shutil.rmtree(f"{OUT}/{pasta}", ignore_errors=True)

## partitionBy: a coluna vira pasta

In [4]:
historico.write.format("delta").mode("overwrite").partitionBy("ano").save(f"{OUT}/partidas_por_ano")

In [5]:
!ls {OUT}/partidas_por_ano | head -5

_delta_log
ano=1930
ano=1934
ano=1938
ano=1950


In [6]:
!ls {OUT}/partidas_por_ano/ano=2018

part-00000-debe2977-ae21-45cb-bfdc-362cb934d502.c000.snappy.parquet


O `partitionBy` grava uma pasta por valor, como `ano=2018`. Na leitura, o nome da pasta volta a ser coluna:

In [7]:
por_ano = spark.read.format("delta").load(f"{OUT}/partidas_por_ano")
por_ano.printSchema()

root
 |-- ano: integer (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- fase: string (nullable = true)
 |-- mandante: string (nullable = true)
 |-- gols_mandante: integer (nullable = true)
 |-- gols_visitante: integer (nullable = true)
 |-- visitante: string (nullable = true)
 |-- publico: integer (nullable = true)
 |-- data_partida: timestamp (nullable = true)
 |-- total_gols: integer (nullable = true)
 |-- resultado: string (nullable = true)



## Partition pruning: não abrir o que não precisa

In [8]:
por_ano.filter(F.col("ano") == 2018).explain(mode="formatted")

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [11]: [match_id#547, fase#548, mandante#549, gols_mandante#550, gols_visitante#551, visitante#552, publico#553, data_partida#554, total_gols#555, resultado#556, ano#546]
Batched: true
Location: PreparedDeltaFileIndex [file:/home/jovyan/work/data/out/partidas_por_ano]
PartitionFilters: [isnotnull(ano#546), (ano#546 = 2018)]
ReadSchema: struct<match_id:int,fase:string,mandante:string,gols_mandante:int,gols_visitante:int,visitante:string,publico:int,data_partida:timestamp,total_gols:int,resultado:string>

(2) ColumnarToRow [codegen id : 1]
Input [11]: [match_id#547, fase#548, mandante#549, gols_mandante#550, gols_visitante#551, visitante#552, publico#553, data_partida#554, total_gols#555, resultado#556, ano#546]

(3) Project [codegen id : 1]
Output [11]: [ano#546, match_id#547, fase#548, mandante#549, gols_mandante#550, gols_visitante#551, visitante#552, publico#553, data_partida#55

Procure **`PartitionFilters`** no plano: com um filtro na coluna de partição, o Spark só abre as pastas que precisa
(**partition pruning**).

In [9]:
por_ano.filter(F.col("mandante") == "Brazil").explain(mode="formatted")

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [11]: [match_id#547, fase#548, mandante#549, gols_mandante#550, gols_visitante#551, visitante#552, publico#553, data_partida#554, total_gols#555, resultado#556, ano#546]
Batched: true
Location: PreparedDeltaFileIndex [file:/home/jovyan/work/data/out/partidas_por_ano]
PushedFilters: [IsNotNull(mandante), EqualTo(mandante,Brazil)]
ReadSchema: struct<match_id:int,fase:string,mandante:string,gols_mandante:int,gols_visitante:int,visitante:string,publico:int,data_partida:timestamp,total_gols:int,resultado:string>

(2) ColumnarToRow [codegen id : 1]
Input [11]: [match_id#547, fase#548, mandante#549, gols_mandante#550, gols_visitante#551, visitante#552, publico#553, data_partida#554, total_gols#555, resultado#556, ano#546]

(3) Filter [codegen id : 1]
Input [11]: [match_id#547, fase#548, mandante#549, gols_mandante#550, gols_visitante#551, visitante#552, publico#553

Fora da partição, o filtro vai para **`PushedFilters`**: o Spark abre todas as pastas.

**Como escolher a coluna:** a que mais aparece nos filtros, com **poucos valores diferentes** (ano, país; não um
identificador).

## Resultado da task 2

Saída por Copa, com partition pruning.

## Para consultar depois

- **Mais de uma coluna:** `partitionBy("ano", "fase")` cria pastas dentro de pastas (`ano=2018/fase=Final`). Cuidado:
  o número de pastas é a combinação dos valores, e cada pasta pode ficar com pouco dado.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [10]:
spark.stop()